---
title: Word Game Hacks/Changes
description: Learn to code Wordle in Javascript and implement core concepts in software with comments
comments: false
layout: post
permalink: /Word_Game/lesson
---

## MAJOR FEATURE: Persistent Leaderboard System
Description
A comprehensive leaderboard system that tracks and displays the top 5 scores across all game sessions. Scores are calculated using a formula that considers WPM, accuracy, and difficulty multipliers to create fair competition across different game modes.

## Leaderboard SubFeatures

The leaderboard system enhances the competitive aspect of the typing game by:

1. Persistent Score Tracking: Maintains top scores across game sessions
2. Difficulty-Based Scoring: Uses multipliers to balance different difficulty levels
3. Comprehensive Metrics: Considers both WPM and accuracy for fair scoring
4. Visual Ranking Display: Shows top 5 performers with difficulty indicators

Scoring Formula: Score = WPM × Difficulty_Multiplier × (Accuracy/100)

This system encourages players to:
- Improve their typing speed and accuracy
- Challenge themselves with higher difficulties for better scores
- Compete against their previous performances
- Track progress over time

The leaderboard updates automatically after each completed game and displays:
- Player rank (1-5)
- WPM achieved
- Accuracy percentage  
- Difficulty level symbol

In [ ]:
// Leaderboard data structure and storage
function loadLeaderboard() {
    // Using in-memory storage since localStorage isn't available
    if (!window.gameLeaderboard) {
        window.gameLeaderboard = [];
    }
    updateLeaderboardDisplay();
}

// Score calculation with difficulty multipliers
function saveScore(wpm, accuracy, difficulty, stringType) {
    const score = {
        wpm: parseInt(wpm),
        accuracy: parseInt(accuracy.replace('%', '')),
        difficulty: difficulty,
        stringType: stringType,
        score: Math.round(parseInt(wpm) * difficulties[difficulty].multiplier * (parseInt(accuracy.replace('%', '')) / 100)),
        timestamp: new Date().toLocaleString()
    };

    if (!window.gameLeaderboard) {
        window.gameLeaderboard = [];
    }
    
    window.gameLeaderboard.push(score);
    window.gameLeaderboard.sort((a, b) => b.score - a.score);
    window.gameLeaderboard = window.gameLeaderboard.slice(0, 5); // Keep top 5
    updateLeaderboardDisplay();
}

// Dynamic leaderboard display updates
function updateLeaderboardDisplay() {
    if (!window.gameLeaderboard || window.gameLeaderboard.length === 0) {
        leaderboardContent.innerHTML = '<div class="leaderboard-entry"><span>No scores yet</span><span>--</span></div>';
        return;
    }

    leaderboardContent.innerHTML = '';
    window.gameLeaderboard.forEach((entry, index) => {
        const div = document.createElement('div');
        div.className = 'leaderboard-entry';
        const diffSymbol = difficulties[entry.difficulty].symbol;
        div.innerHTML = `
            <span>${index + 1}. ${entry.wpm} WPM ${diffSymbol}</span>
            <span>${entry.accuracy}%</span>
        `;
        leaderboardContent.appendChild(div);
    });
}

/* Leaderboard styling */
.leaderboard {
    background: rgba(0, 0, 0, 0.7);
    border-radius: 15px;
    padding: 20px;
    margin: 20px auto;
    width: 300px;
    text-align: center;
    box-shadow: 0 8px 25px rgba(0, 0, 0, 0.3);
}

.leaderboard h3 {
    color: #FFD700;
    margin-bottom: 15px;
    text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.8);
}

.leaderboard-entry {
    display: flex;
    justify-content: space-between;
    padding: 8px 0;
    border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    font-size: 14px;
}

In [ ]:
<!-- HTML structure for leaderboard -->
<div class="leaderboard">
    <h3>🏆 Top Scores</h3>
    <div id="leaderboardContent">
        <div class="leaderboard-entry">
            <span>No scores yet</span>
            <span>--</span>
        </div>
    </div>
</div>

## MINI FEATURE 1: Current Character Highlighting
Description
Visual indicator that highlights the next character to be typed with a yellow background, making it easier for users to see exactly where they are in the text and what character comes next.

## Mini Feature #1 SubFeatures
The current character highlighting system improves user experience by:

1. Visual Guidance: Shows exactly which character to type next
2. Enhanced Focus: Reduces eye strain by clearly indicating current position
3. Better Accuracy: Helps users avoid looking ahead and making mistakes
4. Real-time Updates: Moves dynamically as user types each character

Implementation Details:
- Uses a semi-transparent yellow background highlight
- Positioned precisely behind the current character
- Updates in real-time with each keystroke
- Maintains visibility across different text layouts and line wrapping

This feature particularly helps:
- New typists who need guidance on positioning
- Users practicing accuracy over speed  
- Anyone working with complex text patterns
- Reducing cognitive load during typing sessions

In [ ]:
// Enhanced drawUserText function with character highlighting
function drawUserText(prompt, input) {
    wordCtx.clearRect(0, 0, wordCanvas.width, wordCanvas.height);
    wordCtx.font = '24px Arial';
    wordCtx.textAlign = 'left';

    const maxWidth = wordCanvas.width - 20;
    const lineHeight = 30;
    const lines = wrapText(prompt, maxWidth);
    const startY = (wordCanvas.height - lines.length * lineHeight) / 2;

    lines.forEach((line, lineIndex) => {
        const lineY = startY + lineIndex * lineHeight;
        const lineX = (wordCanvas.width - wordCtx.measureText(line).width) / 2;
        
        let currentX = lineX;
        const startCharIndex = lines.slice(0, lineIndex).join(' ').length + (lineIndex > 0 ? 1 : 0);
        const endCharIndex = startCharIndex + line.length;

        // Draw each character individually for precise control
        for (let i = startCharIndex; i < endCharIndex; i++) {
            const char = prompt[i] || '';
            let color = '#dededeff'; // Default prompt color
            
            if (i < input.length) {
                // Character has been typed
                const typedChar = input[i];
                color = typedChar === char ? '#00ff00' : '#ff0000';
            } else if (i === input.length) {
                // CURRENT CHARACTER HIGHLIGHTING - MINI FEATURE 1
                wordCtx.fillStyle = 'rgba(255, 255, 0, 0.4)';
                wordCtx.fillRect(currentX - 2, lineY - 22, wordCtx.measureText(char).width + 4, 26);
                color = '#ffffff';
            }
            
            wordCtx.fillStyle = color;
            wordCtx.fillText(char, currentX, lineY);
            currentX += wordCtx.measureText(char).width;
        }
    });
}

## MINI FEATURE 2: Advanced Difficulty System
Description
A multi-tier difficulty system that progressively increases challenge through different typing constraints and provides score multipliers to maintain fair competition across difficulty levels.

The difficulty system adds depth and replayability through four distinct modes:

1. Normal Mode (1x multiplier):
   - Standard typing with backspace allowed
   - Case insensitive matching
   - Most forgiving for beginners

2. Hard Mode (1.5x multiplier):  
   - No backspace allowed - mistakes must be overtyped
   - Case insensitive matching
   - Teaches accuracy under pressure

3. Expert Mode (2x multiplier):
   - No backspace allowed
   - Case sensitive matching  
   - Requires precise typing skills

4. Insane Mode (3x multiplier):
   - No backspace allowed
   - Case sensitive matching
   - Game ends immediately on first mistake
   - Ultimate challenge for expert typists

The difficulty system provides:
- Progressive skill development pathway
- Score multipliers to maintain fair competition
- Different typing constraints to improve various skills
- Visual indicators to show current difficulty level

Each mode teaches different typing skills:
- Normal: Basic speed and familiarity
- Hard: Accuracy under pressure (no corrections)
- Expert: Precision typing with case sensitivity
- Insane: Perfect execution under maximum constraints

In [ ]:
// Difficulty configuration system
const difficulties = {
    normal: { 
        name: 'Normal', 
        color: 'easy', 
        symbol: '●', 
        multiplier: 1,
        description: 'Standard typing with backspace allowed'
    },
    hard: { 
        name: 'Hard', 
        color: 'medium', 
        symbol: '◆', 
        multiplier: 1.5,
        description: 'No backspace - mistakes must be overtyped'
    },
    expert: { 
        name: 'Expert', 
        color: 'hard', 
        symbol: '★', 
        multiplier: 2,
        description: 'No backspace + case sensitive'
    },
    insane: { 
        name: 'Insane', 
        color: 'expert', 
        symbol: '⚡', 
        multiplier: 3,
        description: 'No backspace + case sensitive + no mistakes allowed'
    }
};

// Difficulty toggle functionality
function updateDifficultyDisplay() {
    const diff = difficulties[currentDifficulty];
    difficultyToggle.textContent = `Difficulty: ${diff.name}`;
    difficultyIndicator.innerHTML = `<span class="${diff.color}">${diff.symbol}</span> ${diff.name} - ${diff.description}`;
}

difficultyToggle.addEventListener('click', () => {
    const diffKeys = Object.keys(difficulties);
    const currentIndex = diffKeys.indexOf(currentDifficulty);
    currentDifficulty = diffKeys[(currentIndex + 1) % diffKeys.length];
    updateDifficultyDisplay();
});

// Difficulty-based keystroke handling
document.onkeydown = function (e) {
    if (finished) return;

    if (e.key.length === 1 && userInput.length < selectedString.length) {
        const nextChar = selectedString[userInput.length];
        let isCorrect = false;
        
        // Apply difficulty rules
        if (currentDifficulty === 'expert' || currentDifficulty === 'insane') {
            // Case sensitive
            isCorrect = e.key === nextChar;
        } else {
            // Case insensitive like original
            isCorrect = e.key.toLowerCase() === nextChar.toLowerCase();
        }
        
        if (currentDifficulty === 'insane' && !isCorrect) {
            // Insane mode: Game over on first mistake
            alert('Game Over! Insane mode requires perfect accuracy.');
            return;
        }
        
        if (!isCorrect) {
            mistakes++;
        }
        
        userInput += e.key;
    } else if (e.key === 'Backspace' && userInput.length > 0) {
        // Backspace restrictions based on difficulty
        if (currentDifficulty === 'normal') {
            userInput = userInput.slice(0, -1);
        }
        // Hard, Expert, and Insane modes don't allow backspace
    }

    drawUserText(selectedString, userInput);
    updateStats(selectedString, userInput, startTime);

    if (userInput === selectedString) {
        finishGame(selectedString, userInput, startTime);
    }
};

In [ ]:
<!-- HTML structure for difficulty system -->
<button id="difficultyToggle">Difficulty: Normal</button>

<div style="text-align: center;">
    <div class="difficulty-indicator" id="difficultyIndicator">
        <span class="easy">●</span> Normal Mode
    </div>
</div>

## Summary of Changes/Hacks

MAJOR FEATURE - Persistent Leaderboard:
Tracks top 5 scores across sessions
Difficulty-based scoring system
Visual feedback on updates
Rank display in completion alerts

MINI FEATURE 1 - Current Character Highlighting:
Yellow background highlight on next character
Real-time position tracking
Enhanced typing guidance
Cross-platform visual consistency

MINI FEATURE 2 - Advanced Difficulty System:
Four progressive difficulty modes
Backspace restrictions by mode
Case sensitivity options
Score multipliers for fair competition

PRESERVED ORIGINAL FEATURES:
Real-time WPM calculation
Real-time accuracy tracking  
Text wrapping and display
Options menu functionality
Mistake tracking system
Canvas-based rendering